# Step 1 : Read File 

In [ ]:
import pypdf 


def reader (file): 
    file = 'attention_is_all_you_need.pdf'


    object = pypdf.PdfReader(file)
    text = ''
    num_of_pages = object.get_num_pages()
    for page_number in range(num_of_pages):  
        current_page = object.pages[page_number]
        text += current_page.extract_text()


    # Para imprimir los primeros 500 caracteres

    print("\nThese are the first 500 characters: \n",text[:500])

    print("\nThis is paper the number of words the text has:\n", len(text))

    object.close() # Ojo, no se le olvide cerrarlo



These are the first 500 characters: 
 Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 

This is paper the number of words the text has:
 39615


# Step 2: Initializing Model

In [31]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

# Se cargan variables generales

api_key = os.getenv("API_KEY")
model_id = os.getenv("MODELO_ID") # Dato curioso, el model ID, no es un ID, es un nombre 

# Initialize Software Development Kit
client = genai.Client(api_key=api_key)

# Instance Model with correct configuration 
def get_response_gemini(contents, system_instruction=None, stream=False):
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0,
        max_output_tokens=7777
    )
    
    method = client.models.generate_content_stream if stream else client.models.generate_content
    
    return method(
        model=model_id,
        contents=contents,
        config=config
    )

response = get_response_gemini("Did this set up work?")
print(response.text)


To tell you if a setup worked, I need more information! Please tell me:

*   **What was the setup?** Describe what you were trying to do.
*   **What were the components or steps involved?**
*   **What was the intended outcome?** What were you hoping to achieve?
*   **What happened instead?** Did it fail? Did it work partially? Did it do something unexpected?
*   **Are there any error messages or specific details you can provide?**

The more details you give me, the better I can understand and tell you if it worked!


# Step 3: Chat with context

## 3.1 'build_system_prompt' function.

Hever, here we organize a prompt (as you do it with claude) Hahahahaha. But talking seriouly, this function, help us, for the construction of a prompt that includes rol, tone for answering, and ofcourse the context for answering. If questions ask me. 

In [28]:
from langchain_core.prompts import SystemMessagePromptTemplate

def build_system_prompt(text):
    system_template = """
        You are an expert on the paper "Attention Is All You Need".

        You must answer only using information contained in that paper.
        If the user asks something that is not supported by the paper, first clarify that you do not have enough context from the paper.
        Then, if helpful, you may answer using analogies and a slightly funny conversational style.

        Here is the paper content:
        --------------------------------------------------------------------
        {text}
        --------------------------------------------------------------------
        End of paper content.
    """

    system_message_obj = SystemMessagePromptTemplate.from_template(system_template)
    system_prompt_template = system_message_obj.format(text=text)

    print(system_prompt_template)

## 3.2 'Chat_with_document' function

This is a import function, because it calls the order two functions. In this ideas order, we need to talk with professor, because I modified the initializing client chunck, so we can reuse it on chat_with_document function. In my opinion is cleaner. 

In [33]:
from google.genai import types

def chat_with_document(message, history, text):

    system_instructions = build_system_prompt(text).content

    gemini_history = [
        types.Content(
            role="model" if msg["role"] == "assistant" else "user",
            parts=[types.Part(text=msg["content"])]
        ) for msg in history
    ]

    current_message = types.Content(role="user", parts=[types.Part(text=message)])
    
    response_stream = get_response_gemini(
        contents=gemini_history + [current_message],
        system_instruction=system_instructions,
        stream=True
    )

    for chunk in response_stream:
        yield chunk.text

